# Function Calling

In the previous lesson we built a RAG pipeline with `RAGBase.rag()` and saw it fail on typos and ambiguous queries. The pipeline is fixed: search → build prompt → LLM. The LLM is a passenger — it never sees the search results until after the prompt is built, so it can't recover from a bad query.

**An agent puts the LLM in charge.** Instead of running search ourselves, we give the LLM a search *tool*. It decides when to call it and what to search for. If the results are bad, it can try again with a different query — on its own, with no special-case code from us.

The mechanism that makes this possible is **function calling**.

## Setup

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from zoomcamp.ingest import load_faq_data, build_index
from zoomcamp.rag_helper import RAGBase
from sqlitesearch import TextSearchIndex

In [2]:
index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="db/faq.db"
)

index.count()

243

### Search function

We define a top-level `search` function that queries the SQLite index directly. The model will reference it by this name, so we keep the Python function name and the tool name aligned — it makes the dispatch step straightforward later.

In [3]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [4]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

openai_client = OpenAI(api_key=api_key)

## Asking Without Tools

First, let's see what the LLM does without any tools. We send a course-specific question and inspect the answer. The model has no access to our FAQ, so it can only respond from general knowledge — the answer will be vague and unhelpful. This is exactly why we need tools.

In [5]:
# Asking without tools - the model answers from general knowledge
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

print(response.output_text)

Absolutely — in many cases you can still join.

If you’ve just discovered the course, the next step is to check:
- whether registration is still open
- whether the course has already started
- whether late enrollment is allowed

If you want, I can help you figure out the best way to ask by drafting a short message like:

> Hi, I just found out about the course and I’m very interested. Is it still possible to join, even though the course may have already started?

If you share the course name or where you found it, I can help you make the message more specific.


## Defining the Tool and Calling with It

We describe the `search` function to the model as a JSON schema. The model doesn't see our Python code — it only sees this description. LLMs are language-agnostic; under the hood we're making HTTP calls, so the tool is described in JSON rather than Python. The same schema would work from TypeScript or any other language.

The `description` field is the most important part: **the model reads it to decide when to call the function.** The `parameters` block is a JSON Schema for the arguments, and marking `query` as `required` ensures the model always fills it in.

We then resend the same question, but this time include the tool in the request. Instead of a text answer, the response now contains a `tool_calls` entry — the model decided it needs to search the FAQ before it can answer. Notice it doesn't pass the user's question verbatim; it rewrites it into better search keywords.

In [6]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

# Now the model decides to call search instead of answering directly
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

# Should show function_call items in output, not a text answer
for item in response.output:
    print(item.type, getattr(item, "name", ""), getattr(item, "arguments", ""))

function_call search {"query":"Can I join the course if I just discovered it? enrollment eligibility join late"}


## Executing the Function and Sending the Result Back

The model can't run our Python code — it only asked us to. We parse the JSON arguments it provided, call `search`, and serialize the result.

We append two things to the conversation history:

1. **`response.output`** — the model's full output for this turn (including the function call), so it sees its own decision in the next call.
2. **`function_call_output`** — the tool result, linked to the specific call by `call_id`. If the model made multiple function calls in one turn, each result is matched by its own ID.

In [7]:
import json

# Append the model's output (tool call + any reasoning) to history
messages.extend(response.output)

# Dispatch the function call
function_call = next(item for item in response.output if item.type == "function_call")
args = json.loads(function_call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

# Add the tool result linked by call_id
messages.append({
    "type": "function_call_output",
    "call_id": function_call.call_id,
    "output": result_json,
})

In [8]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
 

In [9]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment eligibility join late"}', call_id='call_6VGsS2IIgBhjNBpYWDGSzi1J', name='search', type='function_call', id='fc_064483e4b4af89f7006a397cdb497881a285e54e5615f2eb06', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_6VGsS2IIgBhjNBpYWDGSzi1J',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answ

## Asking the Model Again

We call the API a second time with the full expanded history: the original question, the model's decision to call `search`, and the FAQ results it got back. Now it can produce a proper, course-specific answer.

We have to replay the whole history because LLMs are stateless between API calls. If we sent only the tool result, the model would have no idea what's going on.

In [10]:
# Second call: model now has the original question + its tool call + the FAQ results
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Yes — you can join even if you just discovered the course.

If you want a certificate, make sure to submit your project while submissions are still open.


## Token Usage and Cost

We just made two API calls instead of one. Each call costs money, so it's worth checking how much one tool-using turn actually costs.

The `usage` field on the response gives us token counts — `input_tokens` and `output_tokens` in the Responses API (vs `prompt_tokens` / `completion_tokens` in the Chat Completions API). We multiply by the per-million-token price to get a dollar figure. Note this is only for the *second* call — the first call also has its own cost. Two calls means we pay twice, and the second call is more expensive because we resend the full history as input.

In [11]:
usage = response.usage
print(usage.input_tokens, usage.output_tokens)

def calculate_openai_mini_price(input_tokens, output_tokens):
    # gpt-5.4-mini pricing: $0.15/M input, $0.60/M output
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_openai_mini_price(usage.input_tokens, usage.output_tokens)
print("Total cost: $", round(result["total_cost"], 8))

599 35
Total cost: $ 0.00011085


## Summary

One tool-using turn = two API calls: one for the model to decide to call `search`, one for the model to synthesize the result. The second call carries the full history as input, so its prompt token count grows with each turn. In a multi-step agent loop, prompt tokens accumulate quickly — monitor `usage` on every response.